# 4. modell – Embedding + Bidirectional(GRU)

**Feladat:** a MedPlain angol orvosi mondatainak bináris osztályozása (szakmai = `1`, laikus = `0`).

**Alaparchitektúra:** `Embedding(mask_zero=True) → Bidirectional(GRU(64)) → Dropout(0.3) → Dense(1, sigmoid)`.

A notebook a projekt közös `dl_prep.py` modulját használja. Az adat-előkészítés, a cikkenkénti felosztás, a csak train-en illesztett tokenizer, a padding és a kiértékelés változatlan marad. A validációs eredmények három seed (`42`, `43`, `44`) átlaga alapján hasonlítjuk össze a konfigurációkat.

**Összevetés András BiLSTM-modelljével:** a repó `dl_2_bilstm.ipynb` notebookjában rögzített közös beállításokat használjuk: `EMBED_DIM=100`, `BATCH_SIZE=64`, kétirányú réteg, 64 egység, külső Dropout `0.3`, Adam `1e-3`, EarlyStopping `patience=3`. A GRU belső `dropout` paramétere ettől külön beállítás; az összehasonlító alapkonfigurációban `0.0`.

A teszthalmazt a hiperparaméter-kísérletek nem használják. A végső konfiguráció kiválasztása után egyetlen `dp.evaluate(..., split="test")` hívás méri a teszteredményt.

In [ ]:
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import dl_prep as dp

SEEDS = [42, 43, 44]
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "nincs (CPU)")

data = dp.load_data()
print(data.summary())
print("Teszt-ujjlenyomat:", dp.test_fingerprint(data))

## Adat-előkészítés és padding

A `dp.load_data()` a `train_models.prepare()` függvényen keresztül végzi a tisztítást, majd cikkenként választja le a tesztet, és a train 12%-ából készíti a validációs halmazt. A tokenizer kizárólag a train mondatokon tanul. A sorozatok post-paddingeltek, ezért a `mask_zero=True` maszkolja a padding tokeneket a GRU-ban. Az üres token-sorozatokat a közös előkészítő kód OOV tokennel kezeli.

In [ ]:
lengths = pd.Series([len(s) for s in data.tokenizer.texts_to_sequences(data.train_df.text)])
fig, ax = plt.subplots(figsize=(10, 4))
for label, name, color in [(1, "Szakmai (original)", "#4C72B0"), (0, "Laikus (simplified)", "#55A868")]:
    ax.hist(lengths[data.train_df.label.values == label], bins=50, alpha=0.55, label=name, color=color)
ax.axvline(data.maxlen, color="crimson", ls="--", label=f"maxlen = {data.maxlen} (95. percentilis)")
ax.set_xlabel("Tokenek száma")
ax.set_ylabel("Mondatok száma")
ax.legend()
ax.set_title("Train mondathosszak")
plt.show()
print(f"Levágott train mondatok: {(lengths > data.maxlen).mean():.1%}")

## Modell és kísérleti segédfüggvények

A `gru_dropout` a GRU réteg **bemeneti dropoutja**. A `dropout` ezzel szemben a BiGRU kimenetére helyezett **külön Dropout réteg** aránya. A `spatial_dropout` az Embedding után opcionálisan alkalmazott `SpatialDropout1D` aránya. A három paramétert a kísérletekben elkülönítve változtatjuk.

In [ ]:
DEFAULTS = dict(units=64, gru_dropout=0.0, dropout=0.3, spatial_dropout=0.0)
SEED_RUNS = []


def build_model(units=64, gru_dropout=0.0, dropout=0.3, spatial_dropout=0.0):
    """BiGRU osztályozó; a belső és a külső Dropout külön szabályozható."""
    model_layers = [
        keras.Input(shape=(data.maxlen,), name="token_ids"),
        layers.Embedding(data.vocab_size, dp.EMBED_DIM, mask_zero=True, name="embedding"),
    ]
    if spatial_dropout > 0:
        model_layers.append(layers.SpatialDropout1D(spatial_dropout, name="spatial_dropout"))
    model_layers.append(
        layers.Bidirectional(
            layers.GRU(units, dropout=gru_dropout, recurrent_dropout=0.0),
            name="bidirectional_gru",
        )
    )
    if dropout > 0:
        model_layers.append(layers.Dropout(dropout, name="output_dropout"))
    model_layers.append(layers.Dense(1, activation="sigmoid", name="classification"))
    model = keras.Sequential(model_layers, name="bigru_classifier")
    return dp.compile_model(model)


def run_config(label, seeds=SEEDS, **overrides):
    """Három seed futtatása; a konfigurációt a validációs mutatók átlagával értékeli."""
    params = {**DEFAULTS, **overrides}
    rows, histories = [], {}
    last_model = None
    for seed in seeds:
        dp.set_seeds(seed)
        model = build_model(**params)
        history, seconds = dp.fit(model, data)
        val_result = dp.evaluate(model, data, split="val")
        row = {
            "seed": int(seed),
            "val_acc": float(val_result["accuracy"]),
            "val_f1": float(val_result["f1"]),
            "val_loss": float(min(history.history["val_loss"])),
            "best_epoch": int(dp.best_epoch(history)),
            "epochs": int(len(history.history["loss"])),
            "sec": float(seconds),
        }
        rows.append(row)
        histories[seed] = history
        SEED_RUNS.append({"beállítás": label, **params, **row})
        last_model = model

    run_df = pd.DataFrame(rows)
    summary = {
        "beállítás": label,
        **params,
        "paraméterek": int(last_model.count_params()),
        "val_acc": float(run_df.val_acc.mean()),
        "±acc": float(run_df.val_acc.std(ddof=1)),
        "val_f1": float(run_df.val_f1.mean()),
        "±f1": float(run_df.val_f1.std(ddof=1)),
        "val_loss": float(run_df.val_loss.mean()),
        "legjobb epoch": float(run_df.best_epoch.mean()),
        "idő (s)": float(run_df.sec.mean()),
        "s/epoch": float((run_df.sec / run_df.epochs).mean()),
    }
    print(f"{label:34s} val_acc {summary['val_acc']:.4f} ± {summary['±acc']:.4f} | "
          f"val_f1 {summary['val_f1']:.4f} | idő {summary['idő (s)']:.1f} s")
    return summary, histories[seeds[0]], run_df


SHOW = ["beállítás", "units", "gru_dropout", "dropout", "spatial_dropout", "paraméterek",
        "val_acc", "±acc", "val_f1", "val_loss", "legjobb epoch", "idő (s)"]
FMT = {"paraméterek": "{:,}", "val_acc": "{:.4f}", "±acc": "{:.4f}", "val_f1": "{:.4f}",
       "val_loss": "{:.4f}", "legjobb epoch": "{:.1f}", "idő (s)": "{:.1f}"}
show = lambda frame: display(frame[SHOW].style.format(FMT).hide(axis="index"))

dp.set_seeds(42)
base_model = build_model()
base_model.summary()

## 1. kísérlet – a GRU belső `dropout` paramétere

A GRU egységszáma (64), a BiGRU utáni külön Dropout (0,3) és a SpatialDropout (0,0) változatlan. Csak a GRU saját `dropout` paraméterét változtatjuk.

In [ ]:
exp1, exp1_hist = [], {}
for gru_drop in [0.0, 0.2, 0.4]:
    label = f"GRU dropout={gru_drop}"
    summary, history, _ = run_config(label, units=64, gru_dropout=gru_drop,
                                     dropout=0.3, spatial_dropout=0.0)
    exp1.append(summary)
    exp1_hist[label] = history
exp1 = pd.DataFrame(exp1)
show(exp1)
dp.plot_history(exp1_hist, "BiGRU – belső GRU dropout (külső Dropout=0,3; seed 42)")
plt.show()

## 2. kísérlet – GRU-egységszám

A belső GRU dropout `0.0`, a külön Dropout `0.3`, a SpatialDropout `0.0`. Kizárólag az egységszám változik (32 / 64 / 128), így az eredmény közvetlenül összevethető András BiLSTM egységszám-kísérletével.

In [ ]:
exp2, exp2_hist = [], {}
for units in [32, 64, 128]:
    label = f"BiGRU({units})"
    summary, history, _ = run_config(label, units=units, gru_dropout=0.0,
                                     dropout=0.3, spatial_dropout=0.0)
    exp2.append(summary)
    exp2_hist[label] = history
exp2 = pd.DataFrame(exp2)
show(exp2)
dp.plot_history(exp2_hist, "BiGRU – egységszám (Dropout=0,3; seed 42)")
plt.show()

best_units = int(exp2.sort_values(["val_acc", "val_f1"], ascending=False).iloc[0]["units"])
print("Az egységszám-kísérlet validációs győztese:", best_units)

## 3. kísérlet – `SpatialDropout1D` az Embedding után

Az egységszámot a 2. kísérletben kiválasztott értéken rögzítjük. A GRU belső dropoutja `0.0`, a külön kimeneti Dropout `0.3`; csak a `SpatialDropout1D` arányát változtatjuk. A `0.0` érték a SpatialDropout nélküli kontrollkonfiguráció.

In [ ]:
exp3, exp3_hist = [], {}
for spatial_drop in [0.0, 0.1, 0.2, 0.3]:
    label = f"SpatialDropout1D={spatial_drop}"
    summary, history, _ = run_config(label, units=best_units, gru_dropout=0.0,
                                     dropout=0.3, spatial_dropout=spatial_drop)
    exp3.append(summary)
    exp3_hist[label] = history
exp3 = pd.DataFrame(exp3)
show(exp3)
dp.plot_history(exp3_hist, f"BiGRU({best_units}) – SpatialDropout1D (külső Dropout=0,3; seed 42)")
plt.show()

## LSTM–GRU összehasonlítás – azonos közös konfiguráció

A `results/bilstm.json` fájlból olvassuk be András **64 egységes, Bidirectional, külső Dropout=0,3** konfigurációjának három seeden mért validációs átlagát, tanítási idejét és paraméterszámát. A GRU ugyanezt a konfigurációt az 1. kísérlet `GRU dropout=0.0` sora adja.

A tanítási idő mindkét notebookban a `dp.fit()` által használt `time.perf_counter()` mérés: a `model.fit()` időtartamát méri, a modellépítést/compile-t és a külön validációs predikciót nem. Az idő csak azonos TensorFlow-, eszköz- és környezeti feltételek mellett hasonlítható közvetlenül.

A BiLSTM eredmény-JSON csak a GPU jelenlétét tárolja, az eszköz pontos típusát nem; András notebookjának összefoglalója RTX 4060 Laptop GPU-t nevez meg. A GRU futtatásánál ellenőrizd, hogy ugyanazon a GPU-n fut-e, mert az azonos „GPU=True” önmagában nem garantál azonos hardvert.


In [ ]:
bilstm_path = dp.ROOT / "results" / "bilstm.json"
if not bilstm_path.exists():
    raise FileNotFoundError("Hiányzik a results/bilstm.json; előbb András BiLSTM notebookját kell lefuttatni.")
bilstm_result = json.loads(bilstm_path.read_text(encoding="utf-8"))
lstm_shared = bilstm_result.get("hyperparams", {}).get("shared_setup")
if not lstm_shared:
    raise KeyError("A results/bilstm.json nem tartalmaz hyperparams.shared_setup mezőt.")

# Az exp1 GRU-dropout=0.0 sora azonos a BiLSTM közös 64/0.3 beállításával.
gru_shared_row = exp1[(exp1.units == 64) & (exp1.gru_dropout == 0.0) &
                      (exp1.dropout == 0.3) & (exp1.spatial_dropout == 0.0)].iloc[0]

shared_comparison = pd.DataFrame([
    {"Modell": "Bidirectional LSTM(64)",
     "Paraméterek": int(lstm_shared["params"]),
     "Val accuracy (3 seed átlag)": float(lstm_shared["val_acc_mean"]),
     "Val accuracy szórás": float(lstm_shared["val_acc_std"]),
     "Tanítási idő (s, átlag)": float(lstm_shared["train_seconds_mean"])},
    {"Modell": "Bidirectional GRU(64)",
     "Paraméterek": int(gru_shared_row["paraméterek"]),
     "Val accuracy (3 seed átlag)": float(gru_shared_row["val_acc"]),
     "Val accuracy szórás": float(gru_shared_row["±acc"]),
     "Tanítási idő (s, átlag)": float(gru_shared_row["idő (s)"])},
])
display(shared_comparison.style.format({
    "Paraméterek": "{:,}", "Val accuracy (3 seed átlag)": "{:.4f}",
    "Val accuracy szórás": "{:.4f}", "Tanítási idő (s, átlag)": "{:.2f}"
}).hide(axis="index"))

lstm_env = bilstm_result.get("setup", {})
gpu_now = bool(tf.config.list_physical_devices("GPU"))
print("BiLSTM környezet: TensorFlow", lstm_env.get("tensorflow"), "| GPU:", lstm_env.get("gpu"))
print("BiGRU környezet:  TensorFlow", tf.__version__, "| GPU:", gpu_now,
      "| eszköz:", tf.config.list_physical_devices("GPU") or "CPU")
if lstm_env.get("tensorflow") != tf.__version__ or bool(lstm_env.get("gpu")) != gpu_now:
    print("FIGYELEM: a TensorFlow-verzió vagy a GPU-elérhetőség eltér; a tanítási idők nem tekinthetők teljesen azonos környezetben mértnek.")
else:
    print("A TensorFlow-verzió és a GPU-elérhetőség egyezik; az időmérés módszere mindkét modellnél dp.fit().")
print("Megjegyzés: a BiLSTM notebook RTX 4060 Laptop GPU-t dokumentál; a JSON csak a GPU jelenlétét tárolja, a konkrét típust nem. Ellenőrizd, hogy ez a futtatás ugyanazon a GPU-n történik-e.")

# Egységszám szerinti kiegészítő összehasonlítás az András-notebookban mentett adatokkal.
lstm_sweep = lstm_shared.get("units_sweep", {})
unit_rows = []
for units in [32, 64, 128]:
    if str(units) not in lstm_sweep:
        continue
    gru_row = exp2[exp2.units == units].iloc[0]
    lstm_row = lstm_sweep[str(units)]
    unit_rows.append({
        "Egység": units,
        "LSTM params": int(lstm_row["params"]),
        "GRU params": int(gru_row["paraméterek"]),
        "LSTM val_acc": float(lstm_row["val_acc"]),
        "GRU val_acc": float(gru_row["val_acc"]),
        "LSTM idő (s)": float(lstm_row["sec"]),
        "GRU idő (s)": float(gru_row["idő (s)"]),
    })
unit_comparison = pd.DataFrame(unit_rows)
display(unit_comparison.style.format({
    "LSTM params": "{:,}", "GRU params": "{:,}", "LSTM val_acc": "{:.4f}",
    "GRU val_acc": "{:.4f}", "LSTM idő (s)": "{:.2f}", "GRU idő (s)": "{:.2f}"
}).hide(axis="index"))

## Tanulási görbék – külön Dropout réteg nélkül és vele

Itt 30 epochon át nincs EarlyStopping, hogy látható legyen a túltanulás kialakulása. A két modell azonos egységszámot használ (a validációs egységszám-kísérlet győztese), és mindkettőnél a GRU belső `dropout=0.0`, valamint `SpatialDropout1D=0.0`. **Csak a BiGRU utáni külön Dropout réteg változik:** nincs (`0.0`) vagy `0.3`.

In [ ]:
curves = {}
curve_details = []
for label, output_dropout in [("Dropout réteg nélkül", 0.0), ("Külön Dropout=0,3", 0.3)]:
    dp.set_seeds(42)
    model = build_model(units=best_units, gru_dropout=0.0,
                        dropout=output_dropout, spatial_dropout=0.0)
    history, seconds = dp.fit(model, data, epochs=30, use_early_stopping=False)
    curves[label] = history
    h = history.history
    detail = {
        "modell": label,
        "legjobb_val_loss": float(min(h["val_loss"])),
        "legjobb_epoch": int(dp.best_epoch(history)),
        "utolsó_train_loss": float(h["loss"][-1]),
        "utolsó_val_loss": float(h["val_loss"][-1]),
        "utolsó_train_accuracy": float(h["accuracy"][-1]),
        "utolsó_val_accuracy": float(h["val_accuracy"][-1]),
        "tanítási_idő_másodperc": float(seconds),
    }
    curve_details.append(detail)
    print(detail)

dp.plot_history(curves, f"BiGRU({best_units}) – EarlyStopping nélkül, 30 epoch")
plt.show()

**Értelmezési szempontok:** a túltanulás jele, ha a train loss tovább csökken / train accuracy tovább nő, miközben a validációs loss emelkedik, illetve a train–validation különbség nő. A külön Dropout rendszerint lassítja a train halmazon való illeszkedést; akkor javítja az általánosítást, ha a validációs görbe kedvezőbb és a rés kisebb. A végleges döntést azonban nem ez az egy seed-es, EarlyStopping nélküli ábra, hanem a fenti konfigurációk három seed-es validációs átlaga határozza meg.

## Végső GRU-beállítás kiválasztása

Az összes hiperparaméter-kísérletből a legnagyobb három-seedes átlagos validációs accuracy alapján választunk; egyezésnél a nagyobb átlagos F1, majd az alacsonyabb validációs loss dönt. Az azonos paraméterkombinációt csak egyszer vesszük figyelembe a rangsorolásban.

In [ ]:
all_exp = pd.concat([
    exp1.assign(kísérlet="GRU belső dropout"),
    exp2.assign(kísérlet="GRU egységszám"),
    exp3.assign(kísérlet="SpatialDropout1D"),
], ignore_index=True)
all_exp = all_exp.drop_duplicates(
    subset=["units", "gru_dropout", "dropout", "spatial_dropout"], keep="first")
ranked = all_exp.sort_values(
    ["val_acc", "val_f1", "val_loss"], ascending=[False, False, True]).reset_index(drop=True)
print("Legjobb konfigurációk (validációs rangsor):")
show(ranked.head(8))

best = ranked.iloc[0]
final_params = {
    "units": int(best["units"]),
    "gru_dropout": float(best["gru_dropout"]),
    "dropout": float(best["dropout"]),
    "spatial_dropout": float(best["spatial_dropout"]),
}
print("Kiválasztott végső konfiguráció:", final_params)

## Végső modell – egyszeri tesztkiértékelés

A kiválasztás már lezárult, kizárólag most használjuk a teszthalmazt. A modellt a projekt konvenciójával összhangban `42` seed-del tanítjuk újra; a tesztet pontosan egyszer értékeljük, majd a `bigru` kulccsal mentjük.

In [ ]:
dp.set_seeds(42)
final_model = build_model(**final_params)
final_history, final_seconds = dp.fit(final_model, data)

# A notebook egyetlen tesztkiértékelési hívása – csak a végleges modellen.
test_result = dp.evaluate(final_model, data, split="test")
print(f"Teszt accuracy: {test_result['accuracy']:.4f} | F1: {test_result['f1']:.4f}")
print(f"Epochok: {len(final_history.history['loss'])} (legjobb: {dp.best_epoch(final_history)}) | "
      f"tanítási idő: {final_seconds:.2f} s | paraméterek: {final_model.count_params():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
dp.plot_confusion(test_result["confusion_matrix"], "Embedding + BiGRU\nKonfúziós mátrix (test)", ax=ax)
plt.show()
dp.plot_history(final_history, "Végső BiGRU modell (EarlyStoppinggal)")
plt.show()
final_model.summary()

## Eredmények mentése – `results/bigru.json`

A `shared_setup` mező a 64 egységes, Dropout 0,3-as konfiguráció három-seedes GRU eredményét menti, ugyanazzal a struktúrával, amelyet András a `results/bilstm.json` fájlban használt. A részletes seed-enkénti kísérleti eredmények is bekerülnek a `hyperparams` mezőbe.

In [ ]:
# A közös, LSTM-mel összevethető GRU-konfiguráció: BiGRU(64), GRU dropout=0,
# külön Dropout=0.3, SpatialDropout1D=0. Ezt az exp1-ben mindhárom seeddel mértük.
gru_shared = gru_shared_row
shared_setup = {
    "units": 64,
    "gru_dropout": 0.0,
    "dropout": 0.3,
    "spatial_dropout": 0.0,
    "batch_size": dp.BATCH_SIZE,
    "embed_dim": dp.EMBED_DIM,
    "params": int(gru_shared["paraméterek"]),
    "val_acc_mean": round(float(gru_shared["val_acc"]), 4),
    "val_acc_std": round(float(gru_shared["±acc"]), 4),
    "val_f1_mean": round(float(gru_shared["val_f1"]), 4),
    "train_seconds_mean": round(float(gru_shared["idő (s)"]), 2),
    "sec_per_epoch": round(float(gru_shared["s/epoch"]), 3),
    "seeds": SEEDS,
    "units_sweep": {
        str(int(r.units)): {
            "params": int(r["paraméterek"]),
            "val_acc": round(float(r.val_acc), 4),
            "sec": round(float(r["idő (s)"]), 2),
        }
        for _, r in exp2.iterrows()
    },
}

experiment_summary = all_exp[["kísérlet", "beállítás", "units", "gru_dropout", "dropout",
                               "spatial_dropout", "paraméterek", "val_acc", "±acc", "val_f1",
                               "val_loss", "idő (s)"]].to_dict(orient="records")
experiment_seed_details = pd.DataFrame(SEED_RUNS).to_dict(orient="records")

params_to_save = {
    **final_params,
    "validation_seeds": SEEDS,
    "selection_metric": "mean validation accuracy; tie-break mean F1, then mean validation loss",
    "shared_setup": shared_setup,
    "lstm_shared_setup_reference": lstm_shared,
    "experiments": experiment_summary,
    "experiment_seed_runs": experiment_seed_details,
    "curve_runs": curve_details,
}

path = dp.save_result(
    key="bigru",
    name="Embedding + BiGRU",
    model=final_model,
    history=final_history,
    result=test_result,
    train_seconds=final_seconds,
    data=data,
    params=params_to_save,
    notes=(f"Validáció alapján kiválasztott konfiguráció: {final_params}; "
           f"a hiperparaméter-kísérletek seedjei: {SEEDS}. "
           "A teszthalmazt csak a végleges modellen értékeltük egyszer."),
)
print("Mentve:", path)

# Formátumellenőrzés: a JSON olvasható, a kulcs és a teszt-ujjlenyomat megfelelő.
saved = json.loads(path.read_text(encoding="utf-8"))
assert saved["key"] == "bigru"
assert saved["test_fingerprint"] == dp.test_fingerprint(data)
assert saved["setup"]["maxlen"] == data.maxlen
print("JSON-ellenőrzés sikeres:", path.name, "| key:", saved["key"],
      "| test fingerprint:", saved["test_fingerprint"])

## Összefoglalás

A notebook lefuttatása után az alábbiak alapján vonj le következtetést:

1. **GRU belső dropout:** a külön Dropout `0.3` mellett melyik `gru_dropout` érték adta a legjobb validációs átlagot?
2. **Egységszám:** hogyan változott az accuracy, a paraméterszám és a tanítási idő 32 / 64 / 128 egységnél?
3. **SpatialDropout1D:** javított-e az Embedding utáni térbeli dropout a validációs általánosításon?
4. **Tanulási görbék:** a külön Dropout réteg csökkentette-e a train–validation rést, és hogyan alakult a validációs loss?
5. **LSTM vs. GRU:** a közös 64/0,3 beállítás mellett melyik modell kisebb, gyorsabb és pontosabb? Az időeredményeket csak azonos TensorFlow- és hardverkörnyezet mellett értelmezd.
6. **Teszt:** a végső, validáció alapján választott konfiguráció teszt accuracy/F1 értékét külön kezeld; ezt már ne használd modellválasztásra.

A notebook újrafuttatása `results/bigru.json` fájlt hoz létre. A `dl_prep.py` fájlhoz nem nyúl.